---
Accession Extraction Validation - Evaluation Component

This module handles the evaluation of model predictions against ground truth.
It computes metrics, statistical tests, and generates reports.

Project: AI6129 Pathogen Tracking System
Version: 1.0
---

In [1]:
import json
import re
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Dict, List, Set, Tuple, Optional, Any
import numpy as np
from scipy import stats
import pandas as pd

---
Type Definitions
---

In [ ]:
class Stratum(Enum):
    """Accession density categories matching validation_splits.json"""
    ZERO = "zero"
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"


@dataclass
class GroundTruthDocument:
    """Represents a single document from ground truth data"""
    pmcid: str
    accessions: Set[str]
    stratum: Stratum
    accession_count: int


@dataclass
class PredictionDocument:
    """Represents model predictions for a single document"""
    pmcid: str
    extracted_accessions: Set[str]
    raw_model_output: Optional[str]
    processing_time_ms: Optional[int]


@dataclass
class DocumentMetrics:
    """Computed metrics for a single document"""
    pmcid: str
    stratum: Stratum
    true_positives: int
    false_positives: int
    false_negatives: int
    precision: float
    recall: float
    f1: float
    is_exact_match: bool
    has_hallucination: bool


@dataclass
class AggregateMetrics:
    """Aggregated metrics over a collection of documents"""
    document_count: int
    macro_precision: float
    macro_recall: float
    macro_f1: float
    micro_precision: float
    micro_recall: float
    micro_f1: float
    exact_match_rate: float
    hallucination_rate: float
    total_true_positives: int
    total_false_positives: int
    total_false_negatives: int


@dataclass
class ConfidenceInterval:
    """Bootstrap confidence interval for a metric"""
    point_estimate: float
    lower_bound: float
    upper_bound: float
    confidence_level: float


@dataclass
class McNemarResult:
    """Result of McNemar's test comparing two models"""
    model_a: str
    model_b: str
    test_type: str
    n_both_correct: int
    n_both_wrong: int
    n_a_only_correct: int
    n_b_only_correct: int
    statistic: float
    p_value: float
    is_significant: bool
    adjusted_alpha: float


@dataclass
class StratifiedReport:
    """Metrics for a single stratum"""
    stratum: Stratum
    metrics: AggregateMetrics
    document_count: int


@dataclass
class EvaluationReport:
    """Complete evaluation report for a single model"""
    model_id: str
    split_name: str
    overall_metrics: AggregateMetrics
    positive_only_metrics: AggregateMetrics
    stratified_reports: List[StratifiedReport]
    confidence_intervals: Dict[str, ConfidenceInterval]
    document_metrics: List[DocumentMetrics]

---
NORMALISATION FUNCTIONS
---

In [ ]:
def normalise_accession(accession: str) -> str:
    """
    Normalise a single accession number for consistent comparison.

    Parameters:
        accession: Raw accession string

    Returns:
        Normalised accession (uppercase, trimmed, version removed)
    """
    normalised = accession.strip().upper()
    normalised = re.sub(r'\.\d+$', '', normalised)
    return normalised


def normalise_accession_set(accessions: List[str]) -> Set[str]:
    """
    Normalise a list of accessions to a set.

    Parameters:
        accessions: List of raw accession strings

    Returns:
        Set of normalised accessions
    """
    normalised_set = set()

    for acc in accessions:
        if acc is None:
            continue
        normalised = normalise_accession(acc)
        if normalised:
            normalised_set.add(normalised)

    return normalised_set

---
Data Loading Functions
---

In [ ]:
def load_ground_truth(ground_truth_filepath: str, split_name: str
                      ) -> Dict[str, GroundTruthDocument]:
    """
    Load ground truth data for a specific split.

    Parameters:
        ground_truth_filepath: Path to validation_splits.json
        split_name: Name of split to load (development/validation/test)

    Returns:
        Mapping of PMCID -> GroundTruthDocument
    """
    ground_truth_docs = {}

    with open(ground_truth_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    split_data = data['splits'].get(split_name, {})
    split_pmcids = split_data.get('files', []) if instance(split_data, dict) else split_data

    document_details = data.get('document_details', {})

    for pmcid in split_pmcids:
        if pmcid  not in document_details:
            continue

        doc_info = document_details[pmcid]
        accessions_raw = doc_info.get('accessions', [])
        accessions_normalised = normalise_accession_set(accessions_raw)
        stratum_str = doc_info.get('stratum', 'zero')
        stratum = Stratum(stratum_str)
        accession_count = doc_info.get('accession_count', 0)

        ground_truth_docs[pmcid] = GroundTruthDocument(
            pmcid=pmcid,
            accessions=accessions_normalised,
            stratum=stratum,
            accession_count=accession_count
        )

    return ground_truth_docs

def load_predictions(predictions_filepath: str
                     ) -> Tuple[Dict[str, PredictionDocument], Dict[str, Any]]:
    """
    Load model predictions from experiment output file.

    Parameters:
        predictions_filepath: Path to predictions JSON file

    Returns:
        Tuple of (PMCID -> PredictionDocument mapping, experiment metadata)
    """
    prediction_docs = {}

    with open(predictions_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    metadata = data.get('experiment_metadata', {})
    predictions_list = data.get('predictions', [])

    for pred in predictions_list:
        pmcid = pred.get('pmcid', '')
        if not pmcid:
            continue

        accessions_raw = pred.get('extraction_accessions', [])
        accessions_normalised = normalise_accession_set(accessions_raw)
        raw_output = pred.get('raw_model_output')
        processing_time = pred.get('processing_time_ms')

        prediction_docs[pmcid] = PredictionDocument(
            pmcid=pmcid,
            extracted_accessions=accessions_normalised,
            raw_model_output=raw_output,
            processing_time_ms=processing_time
        )

    return prediction_docs, metadata

def load_split_pmcids(ground_truth_filepath: str,
                      split_name: str
                      ) -> List[str]:
    """
    Get list of PMCIDs for a specific split.

    Parameters:
        ground_truth_filepath: Path to validation_splits.json
        split_name: Name of split

    Returns:
        List of PMCIDs in the split
    """
    with open(ground_truth_filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)

    split_data = data['splits'].get(split_name, {})
    return split_data.get('files', []) if instance(split_data, dict) else split_data



---
metric computation function
---

In [ ]:
def compute_set_overlap(
    predicted: Set[str],
    ground_truth: Set[str]
) -> Tuple[int, int, int]:
    """
    Calculate set overlap statistics.

    Parameters:
        predicted: Set of predicted accessions
        ground_truth: Set of ground truth accessions

    Returns:
        Tuple of (true_positives, false_positives, false_negatives)
    """
    true_positives = len(predicted & ground_truth)
    false_positives = len(predicted - ground_truth)
    false_negatives = len(ground_truth - predicted)

    return true_positives, false_positives, false_negatives


def compute_precision(true_positives: float, false_positives: float) -> float:
    """Calculate precision: TP / (TP + FP)"""
    denominator = true_positives + false_positives
    if denominator == 0:
        return 1.0
    return true_positives / denominator


def compute_recall(true_positives: float, false_negatives: float) -> float:
    """Calculate recall: TP / (TP + FN)"""
    denominator = true_positives + false_negatives
    if denominator == 0:
        return 1.0
    return true_positives / denominator


def compute_f1(precision: float, recall: float) -> float:
    """Calculate F1 score: harmonic mean of precision and recall"""
    if precision + recall == 0:
        return 0.0
    return 2 * (precision * recall) / (precision + recall)

def compute_document_metrics(
    prediction: PredictionDocument,
    ground_truth: GroundTruthDocument
) -> DocumentMetrics:
    """
    Compute all metrics for a single document.

    Parameters:
        prediction: Model prediction for document
        ground_truth: Ground truth for document

    Returns:
        DocumentMetrics with all computed values
    """
    tp, fp, fn = compute_set_overlap(
        prediction.extracted_accessions,
        ground_truth.accessions
    )

    precision = compute_precision(tp, fp)
    recall = compute_recall(tp, fn)
    f1 = compute_f1(precision, recall)

    is_exact = (prediction.extracted_accessions == ground_truth.accessions)
    has_hallucination = (fp > 0)

    return DocumentMetrics(
        pmcid=ground_truth.pmcid,
        stratum=ground_truth.stratum,
        true_positives=tp,
        false_positives=fp,
        false_negatives=fn,
        precision=precision,
        recall=recall,
        f1=f1,
        is_exact_match=is_exact,
        has_hallucination=has_hallucination
    )

def compute_aggregate_metrics(
    document_metrics_list: List[DocumentMetrics]
) -> AggregateMetrics:
    """
    Aggregate metrics across multiple documents.

    Parameters:
        document_metrics_list: List of per-document metrics

    Returns:
        AggregateMetrics with macro and micro averages
    """
    if not document_metrics_list:
        return AggregateMetrics(
            document_count=0,
            macro_precision=0.0,
            macro_recall=0.0,
            macro_f1=0.0,
            micro_precision=0.0,
            micro_recall=0.0,
            micro_f1=0.0,
            exact_match_rate=0.0,
            hallucination_rate=0.0,
            total_true_positives=0,
            total_false_positives=0,
            total_false_negatives=0
        )

    document_count = len(document_metrics_list)

    #Macro averags
    macro_precision = sum(m.precision for m in document_metrics_list) / document_count
    macro_recall = sum(m.recal for m in document_metrics_list) / document_count
    macro_f1 = sum(m.f1 for m in document_metrics_list)

    #Totals for micro averages
    total_tp = sum(m.precision for m in document_metrics_list)
    total_fp = sum(m.recall for m in document_metrics_list)
    total_fn = sum(m.false_negatives for m in document_metrics_list)

    #micro averages
    micro_precision = compute_precision(total_tp, total_fp)
    micro_recall = compute_recall(total_tp, total_fn)
    micro_f1 = compute_f1(micro_precision, micro_recall)

    #Rates
    exact_match_count = sum(1 for m in document_metrics_list if m.is_exact_match)
    hallucination_count = sum(1 for m in document_metrics_list if m.has_hallucination)

    exact_match_rate = sum(1 for m in document_metrics_list if m.is_exact_match)
    hallucination_rate = usm(1 for m in document_metrics_list if m.has_hallucination)

    return AggregateMetrics(
        document_count=document_count,
        macro_precision=macro_precision,
        macro_recall=macro_recall,
        macro_f1=macro_f1,
        micro_precision=micro_precision,
        micro_recall=micro_recall,
        micro_f1=micro_f1,
        exact_match_rate= exact_match_rate,
        hallucination_rate=hallucination_rate,
        total_true_positives=total_tp,
        total_false_positives=total_fp,
        total_false_negatives=total_fn
        )

def filter_positive_documents(document_metrics_list: List[DocumentMetrics],
                              ground_truth_docs: Dict[str, GroundTruthDocument]
                              ) -> List[DocumentMetrics]:
    """
    Filter to documents that have at least one ground truth accession.

    Parameters:
        document_metrics_list: All document metrics
        ground_truth_docs: Ground truth lookup

    Returns:
        Filtered list with only positive documents
    """
    positive_docs = []

    for doc_metrics in document_metrics_list:
        gt_doc = ground_truth_docs.get(doc_metrics.pmcid)
        if gt_doc and len(gt_doc.accessions) > 0:
            positive_docs.append(doc_metrics)

    return positive_docs

def filter_by_stratum(
    document_metrics_list: List[DocumentMetrics],
    stratum: Stratum
) -> List[DocumentMetrics]:
    """
    Filter documents by density stratum.

    Parameters:
        document_metrics_list: All document metrics
        stratum: Target stratum to filter for

    Returns:
        Filtered list with only matching stratum
    """
    return [m for m in document_metrics_list if m.stratum == stratum]

def compute_stratified_metrics(document_metrics_list: List[DocumentMetrics]
                               ) -> List[StratifiedReport]:
    """
    Compute metrics for each density stratum.

    Parameters:
        document_metrics_list: All document metrics

    Returns:
        List of StratifiedReport for each stratum
    """
    reports = []

    for layer in Stratum:
        stratum_docs = filter_by_stratum(document_metrics_list, layer)

        if not stratum_docs:
            continue

        metrics = compute_aggregate_metrics(stratum_docs)
        reports.append(StratifiedReport(
            stratum=layer,
            metrics=metrics,
            document_count=len(stratum_docs)
        ))
    return reports

---
STATISTICAL TESTS
---

In [ ]:
def build_mcnemar_contingency_table(metrics_a: List[DocumentMetrics],
                                    metrics_b: List[DocumentMetrics],
                                    test_type: str
                                    ) -> Tuple[int, int, int, int]:
    """
    Build 2x2 contingency table for McNemar's test.

    Parameters:
        metrics_a: Document metrics from model A
        metrics_b: Document metrics from model B
        test_type: "exact_match" or "hallucination"

    Returns:
        Tuple of (both_correct, both_wrong, a_only_correct, b_only_correct)
    """
    metrics_a_dict = {m.pmcid: m for m in metrics_a}
    metrics_b_dict = {m.pmcid: m for m in metrics_b}

    common_pmcids = set(metrics_a_dict.keys()) & set(metrics_b_dict.keys())

    both_correct = 0
    both_wrong = 0
    a_correct = 0
    b_correct = 0

    for pmcid in common_pmcids:
        m_a = metrics_a_dict[pmcid]
        m_b = metrics_b_dict[pmcid]

        if test_type == "exact_match":
            a_only = m_a.is_exact_match
            b_only = m_b.is_exact_match
        else:
            a_only = not m_a.has_hallucination
            b_only = not m_b.has_hallucination

        if a_only and b_only:
            both_correct += 1
        elif not a_only and b_only:
            both_wrong += 1
        elif a_correct and not b_correct:
            a_correct += 1
        else:
            b_correct += 1

    return both_correct, both_wrong, a_correct, b_correct


def compute_mcnemar_test(
    metrics_a: List[DocumentMetrics],
    metrics_b: List[DocumentMetrics],
    model_a_id: str,
    model_b_id: str,
    test_type: str,
    alpha: float = 0.05
) -> McNemarResult:
    """
    Perform McNemar's test comparing two models.

    Parameters:
        metrics_a: Document metrics from model A
        metrics_b: Document metrics from model B
        model_a_id: Identifier for model A
        model_b_id: Identifier for model B
        test_type: "exact_match" or "hallucination"
        alpha: Significance level

    Returns:
        McNemarResult with test statistics
    """
    both_correct, both_wrong, a_only, b_only = build_mcnemar_contingency_table(metrics_a, metrics_b, test_type)
    n_discordant = a_only + b_only

    if n_discordant == 0:
        statistic = 0.0
        p_value = 1.0
    elif n_discordant < 25:
        result = stats.binomtest(a_only, n_discordant, 0.5)
        statistic = float(a_only)
        p_value = result.pvalue
    else:
        statistic = ((abs(a_only - b_only) - 1) ** 2) / (a_only + b_only)
        p_value = 1 - stats.chi2.cdf(statistic, df=1)

    is_significant = p_value < alpha

    return McNemarResult(
        model_a=model_a_id,
        model_b=model_b_id,
        test_type=test_type,
        n_both_correct=both_correct,
        n_both_wrong=both_wrong,
        n_a_only_correct=a_only,
        n_b_only_correct=b_only,
        statistic=statistic,
        p_value=p_value,
        is_significant=is_significant,
        adjusted_alpha=alpha
    )


def apply_holm_bonferroni_correction(
    results: List[McNemarResult],
    alpha: float = 0.05
) -> List[McNemarResult]:
    """
    Apply Holm-Bonferroni correction for multiple comparisons.

    Parameters:
        results: List of McNemar test results
        alpha: Family-wise error rate

    Returns:
        Updated results with corrected significance
    """
    if not results:
        return results

    n_tests = len(results)
    sorted_results = sorted(results, key=lambda r:r.p_value)

    corrected_results = []

    for i, result in enumerate(sorted_results):
        adjusted_alpha = alpha / (n_tests - 1)
        is_significant = result. p_value < adjusted_alpha

        corrected_result = McNemarResult(
            model_a=result.model_a,
            model_b=result.model_b,
            test_type=result.test_type,
            n_both_correct=result.n_both_correct,
            n_both_wrong=result.n_both_wrong,
            n_a_only_correct=result.n_a_only_correct,
            n_b_only_correct=result.n_b_only_correct,
            statistic=result.statistic,
            p_value=result.p_value,
            is_significant=is_significant,
            adjusted_alpha=adjusted_alpha
        )
        corrected_results.append(corrected_result)

    return corrected_results

---
BOOTSTRAP CONFIDENCE INTERVALS
---

In [ ]:
def bootstrap_metric(document_metrics_list: List[DocumentMetrics],
                     metric_fn: callable,
                     n_bootstrap: int = 1000,
                     confidence_level: float = 0.95,
                     random_seed: int = 42
                    ) -> ConfidenceInterval:
    """
    Compute bootstrap confidence interval for a metric.

    Parameters:
        document_metrics_list: List of document metrics
        metric_fn: Function that computes metric from list of DocumentMetrics
        n_bootstrap: Number of bootstrap samples
        confidence_level: Confidence level (e.g., 0.95)
        random_seed: Random seed for reproducibility

    Returns:
        ConfidenceInterval with bounds
    """
    np.random.seed(random.seed)

    if n.docs == 0:
        return ConfidenceInterval(
            point_estimate=0.0,
            lower_bound=0.0,
            upper_bound=0.0,
            confidence_level=confidence_level
        )
    point_estimate = metric_fn(document_metrics_list)

    bootstrap_values = []

    for _ in range(n_bootstrap):
        indices = np.random.radint(0, n_docs, size=n_docs)
        sample = [document_metrics_list[i] for i in indices]
        bootstrap_values.append(metric_fn(sample))

    alpha = 1 - confidence_level
    lower_percentile = (alpha / 2)*100
    upper_percentile = (1-alpha / 2) * 100

    lower_bound = float(np.percentile(bootstrap_values, lower_percentile))
    upper_bound = float(np.percentile(bootstrap_values, upper_percentile))

    return ConfidenceInterval(
        point_estimate=point_estimate,
        lower_bound=lower_bound,
        upper_bound=upper_bound,
        confidence_level=confidence_level
    )



def compute_confidence_intervals(document_metrics_list: List[DocumentMetrics],
                                 n_bootstrap: int = 1000,
                                 confidence_level: float = 0.95
                                ) -> Dict[str, ConfidenceInterval]:
    """
    Compute confidence intervals for key metrics.

    Parameters:
        document_metrics_list: List of document metrics
        n_bootstrap: Number of bootstrap samples
        confidence_level: Confidence level

    Returns:
        Dictionary mapping metric name to ConfidenceInterval
    """
    intervals = {}

    # Macro F1
    def macro_f1_fn(docs):
        if not docs:
            return 0.0
        return sum(m.f1 for m in docs) / len(docs)

    intervals['macro_f1'] = bootstrap_metric(
        document_metrics_list, macro_f1_fn, n_bootstrap, confidence_level
        )

    # Exact match rate
    def exact_match_fn(docs):
        if not docs:
            return 0.0
        return sum(1 for m in docs if m.is_exact_match) / len(docs)

    intervals['exact_match_rate'] = bootstrap_metric(
        document_metrics_list, exact_match_fn, n_bootstrap, confidence_level
    )

    # Hallucination rate
    def hallucination_fn(docs):
        if not docs:
            return 0.0
        return sum(1 for m in docs if m.has_hallucination) / len(docs)

    intervals['hallucination_rate'] = bootstrap_metric(
        document_metrics_list, hallucination_fn, n_bootstrap, confidence_level
    )

    return intervals

---
Main Evaluation Function
---

In [ ]:
def evaluate_single_model(predictions_filepath: str,
                          ground_truth_filepath: str,
                          split_name: str
                        ) -> EvaluationReport:
    """
    Perform full evaluation for a single model.

    Parameters:
        predictions_filepath: Path to model predictions
        ground_truth_filepath: Path to ground truth
        split_name: Data split name

    Returns:
        Complete EvaluationReport
    """

    predictions, metadata = load_predictions(predictions_filepath)
    ground_truth_docs = load_ground_truth(ground_truth_filepath, split_name)

    model_id = metadata.get('model_id', 'unknown')

    #compute per-document metrics
    document_metrics_list = []

    for pmcid, gt_doc in ground_truth_docs.items():
        if pmcid not in predictions:
            pred_doc = PredictionDocument(
                pmcid=pmcid,
                extracted_accessions=set(),
                raw_model_output=None,
                processing_time_ms=None
            )
        else:
            pred_doc = predictions[pmcid]

        doc_metrics = compute_document_metrics(pred_doc, gt_doc)
        document_metrics_list.append(doc_metrics)

    #overall metrics
    overall_metrics = compute_aggregate_metrics(document_metrics_list)

    #positive-only metrics
    positive_docs = filter_positive_documents(document_metrics_list, ground_truth_docs)
    positive_only_metrics = compute_aggregate_metrics(positive_docs)

    #Stratified metrics
    stratified_reports = compute_stratified_metrics(document_metrics_list)

    #confidence intervals
    confidence_intervals = compute_confidence_intervals(document_metrics_list)

    return EvaluationReport(
        model_id=model_id,
        split_name=split_name,
        overall_metrics=overall_metrics,
        positive_only_metrics=positive_only_metrics,
        stratified_reports=stratified_reports,
        confidence_intervals=confidence_intervals,
        document_metrics=document_metrics_list
    )

def compare_models(reports: List[EvaluationReport],
                   alpha: float = 0.05
                ) -> List[McNemarResult]:
    """
    Perform pairwise model comparisons using McNemar's test.

    Parameters:
        reports: List of evaluation reports for different models
        alpha: Family-wise error rate

    Returns:
        List of McNemar test results with Holm-Bonferroni correction
    """
    all_results = []

    for i in range(len(reports)):
        for j in range(i+1, len(reports)):
            report_a = reports[i]
            report_b = reports[j]

            #Exact match comparison
            em_result = compute_mcnemar_test(
                report_a.document_metrics,
                report_b.document_metrics,
                report_a.model_id,
                report_b.model_id,
                "exact_match",
                alpha
            )
            all_results.append(em_result)

            #Hallucianation comparison

            hal_result = compute_mcnemar_test(
                report_a.document_metrics,
                report_b.document_metrics,
                report_a.model_id,
                report_b.model_id,
                "hallucination",
                alpha
            )
            all_results.append(hal_result)

    corrected_results= apply_holm_bonferroni_correction(all_results, alpha)

    return corrected_results

---
Report Generation
---

In [ ]:
def generate_summary_table(reports: List[EvaluationReport]
                           ) -> pd.DataFrame:
    """
    Generate summary table comparing all models.

    Parameters:
        reports: List of evaluation reports

    Returns:
        DataFrame with model comparison
    """
    rows = []

    for report in reports:
        row = {
            'model_id': report.model_id,
            'split': report.split_name,
            'n_documents': report.overall_metrics.document_count,
            'macro_precision': report.overall_metrics.macro_precision,
            'macro_recall': report.overall_metrics.macro_recall,
            'macro_f1': report.overall_metrics.macro_f1,
            'micro_f1': report.overall_metrics.micro_f1,
            'exact_match_rate': report.overall_metrics.exact_match_rate,
            'hallucination_rate': report.overall_metrics.hallucination_rate,
            'f1_ci_lower': report.confidence_intervals['macro_f1'].lower_bound,
            'f1_ci_upper': report.confidence_intervals['macro_f1'].upper_bound
        }
        rows.append(row)

    return pd.DataFrame(rows)

def generate_stratified_table(reports: List[EvaluationReport]
                              ) -> pd.DataFrame:
    """
    Generate stratified performance table.

    Parameters:
        reports: List of evaluation reports

    Returns:
        DataFrame with stratified metrics
    """
    rows = []

    for report in reports:
        for strat_report in report.stratified_reports:
            row ={
                'model_id': report.model_id,
                'stratum': strat_report.stratum.value,
                'n_documents': strat_report.document_count,
                'macro_f1': strat_report.metrics.macro_f1,
                'exact_match_rate': strat_report.metrics.exact_match_rate,
                'hallucination_rate': strat_report.metrics.hallucination_rate
            }
            rows.append(row)

    return pd.DataFrame(rows)

def generate_mcnemar_table(results: List[McNemarResult]
                           ) -> pd.DataFrame:
    """
    Generate table of McNemar test results.

    Parameters:
        results: List of McNemar test results

    Returns:
        DataFrame with statistical comparisons
    """
    rows = []

    for result in results:
        row = {
            'model_a': result.model_a,
            'model_b': result.model_b,
            'test_type': result.test_type,
            'n_discordant': result.n_a_only_correct + result.n_b_only_correct,
            'a_only_correct': result.n_a_only_correct,
            'b_only_correct': result.n_b_only_correct,
            'statistic': result.statistic,
            'p_value': result.p_value,
            'adjusted_alpha': result.adjusted_alpha,
            'is_significant': result.is_significant
        }
        rows.append(row)

    return pd.DataFrame(rows)

def save_evaluation_report(
    report: EvaluationReport,
    output_filepath: str
) -> None:
    """
    Save evaluation report to JSON file.

    Parameters:
        report: EvaluationReport to save
        output_filepath: Path for output file
    """
    def metrics_to_dict(m: AggregateMetrics) -> dict:
        return {
            'document_count': m.document_count,
            'macro_precision': m.macro_precision,
            'macro_recall': m.macro_recall,
            'macro_f1': m.macro_f1,
            'micro_precision': m.micro_precision,
            'micro_recall': m.micro_recall,
            'micro_f1': m.micro_f1,
            'exact_match_rate': m.exact_match_rate,
            'hallucination_rate': m.hallucination_rate,
            'total_true_positives': m.total_true_positives,
            'total_false_positives': m.total_false_positives,
            'total_false_negatives': m.total_false_negatives
        }

    def ci_to_dict(ci: ConfidenceInterval) -> dict:
        return {
            'point_estimate': ci.point_estimate,
            'lower_bound': ci.lower_bound,
            'upper_bound': ci.upper_bound,
            'confidence_level': ci.confidence_level
        }

    output_data = {
        'model_id': report.model_id,
        'split_name': report.split_name,
        'overall_metrics': metrics_to_dict(report.overall_metrics),
        'positive_only_metrics': metrics_to_dict(report.positive_only_metrics),
        'stratified_reports':[{
                'stratum': sr.stratum.value,
                'document_count': sr.document_count,
                'metrics': metrics_to_dict(sr.metrics)
        }
        for sr in report.stratified_reports
        ],
        'confidence_intervals':{
            name: ci_to_dict(ci) for name, ci in report.confidence_intervals.items()
        }
    }

    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(output_data, f, indent=2)

---
Main Entry Point
---